In [ ]:
import pdfplumber
import re
import pandas as pd

def extraer_fino(pdf_path):
    movimientos = []
    # Expresión regular para: Fecha (00/00) + Referencia (opcional) + El resto
    patron_fila = re.compile(r'^(\d{2}/\d{2})\s+(\d*)\s+(.*)\s+([\d,.]+-?)$')
    inicio_patron="ARCENTALES"
    fin_patron="SUBTOTAL :"

    with pdfplumber.open(pdf_path) as pdf:
        # Los movimientos suelen estar en la página 3 o 4
        pagina = pdf.pages[2] 
        texto = pagina.extract_text().split("\n")
        print("TEXTO")
        print(texto)
        idx_inicio = next((i for i, s in enumerate(texto) if inicio_patron.lower() in s.lower()), None) +1
        idx_final = next((i for i, s in enumerate(texto) if fin_patron.lower() in s.lower()), None)
        print("Inicio:",idx_inicio)
        print("Final:",idx_final)
        print(texto)
        table = texto[idx_inicio:idx_final]
        for linea in table:
            print("LINEA")
            print(linea)
            match = patron_fila.match(linea)
            # if match:
            fecha, ref, desc, valor = match.groups()
            
            # Arreglar el formato del número (Pichincha style)
            valor = valor.replace('.', '').replace(',', '.')
            if valor.endswith('-'):
                valor = '-' + valor.replace('-', '')
            
            movimientos.append([fecha, ref, desc, float(valor)])

    return pd.DataFrame(movimientos, columns=["Fecha", "Ref", "Descripción", "Monto"])

def extraer_totales_pichincha(pdf_path):
    totales = {
        "saldo_anterior": 0.0,
        "subtotal_pagos": 0.0,
        "total_consumos_mes": 0.0,
        "total_a_pagar": 0.0,
        "pago_minimo": 0.0
    }
    
    with pdfplumber.open(pdf_path) as pdf:
        # Extraemos todo el texto para buscar patrones globales
        texto_completo = ""
        for page in pdf.pages:
            texto_completo += page.extract_text() + "\n"
            
        # Diccionario de patrones basados en la estructura del PDF
        # Manejamos el formato 1.234,56 -> 1234.56
        patrones = {
            "saldo_anterior": r"SALDO ANTERIOR\s+([\d,.]+)",
            "subtotal_pagos": r"SUBTOTAL PAGOS\s+([\d,.]+)",
            "total_consumos_mes": r"TOTAL CONSUMOS MES\s+([\d,.]+)",
            "total_a_pagar": r"TOTAL A PAGAR\s+([\d,.]+)",
            "pago_minimo": r"MINIMO A PAGAR\s+([\d,.]+)"
        }
        
        for clave, pattern in patrones.items():
            match = re.search(pattern, texto_completo, re.IGNORECASE)
            if match:
                # Limpieza: quitamos el punto de miles y cambiamos la coma decimal por punto
                valor_str = match.group(1).replace('.', '').replace(',', '.')
                totales[clave] = float(valor_str)
                
    return totales


In [ ]:
import pdfplumber
import pandas as pd
import re
import os

def procesar_estado_pichincha_completo(file_path):
    file_name = os.path.basename(file_path)
    
    with pdfplumber.open(file_path) as pdf:
        # Extraemos el texto completo para el parsing de metadata
        texto_completo = "\n".join([p.extract_text() for p in pdf.pages if p.extract_text()])
        
        # --- 1. FUNCIÓN INTERNA DE LIMPIEZA ---
        def limpiar_monto(texto):
            if not texto: return 0.0
            # Normalización: 1.234,56 -> 1234.56
            num = texto.replace('.', '').replace(',', '.')
            if num.endswith('-'):
                num = '-' + num.replace('-', '')
            try:
                return float(num)
            except ValueError:
                return 0.0

        # --- 2. EXTRACCIÓN DE METADATA CON REGEX ---
        # Buscamos los valores clave en el texto del PDF
        meta_raw = {
            'EMPRESA':          "BANCO PICHINCHA",
            'NUM_TARJETA':      re.search(r'(\d{4}X+5647)', texto_completo).group(1) if re.search(r'(\d{4}X+5647)', texto_completo) else "N/A",
            'FECHA_EMISION':    re.search(r'FECHA DE EMISIÓN:\s+([\w\s]+)', texto_completo).group(1).strip() if re.search(r'FECHA DE EMISIÓN:\s+([\w\s]+)', texto_completo) else None,
            'FECHA_MAX_PAGO':   re.search(r'FECHA MÁXIMA DE PAGO SIN RECARGOS\s+([\w\s]+)', texto_completo).group(1).strip() if re.search(r'FECHA MÁXIMA DE PAGO SIN RECARGOS\s+([\w\s]+)', texto_completo) else None,
            'SALDO_ANTERIOR':   limpiar_monto(re.search(r'SALDO ANTERIOR\s+([\d,.]+)', texto_completo).group(1)) if re.search(r'SALDO ANTERIOR\s+([\d,.]+)', texto_completo) else 0.0,
            'SUBTOTAL_PAGADO':  limpiar_monto(re.search(r'SUBTOTAL PAGOS\s+([\d,.]+)', texto_completo).group(1)) if re.search(r'SUBTOTAL PAGOS\s+([\d,.]+)', texto_completo) else 0.0,
            'TOTAL_A_PAGAR':    limpiar_monto(re.search(r'TOTAL A PAGAR\s+([\d,.]+)', texto_completo).group(1)) if re.search(r'TOTAL A PAGAR\s+([\d,.]+)', texto_completo) else 0.0,
            'MINIMO_A_PAGAR':   limpiar_monto(re.search(r'MINIMO A PAGAR\s+([\d,.]+)', texto_completo).group(1)) if re.search(r'MINIMO A PAGAR\s+([\d,.]+)', texto_completo) else 0.0,
            'TOTAL_CONSUMO':    limpiar_monto(re.search(r'TOTAL CONSUMOS MES\s+([\d,.]+)', texto_completo).group(1)) if re.search(r'TOTAL CONSUMOS MES\s+([\d,.]+)', texto_completo) else 0.0,
        }

        # --- 3. EXTRACCIÓN DE MOVIMIENTOS ---
        transacciones = []
        patron_fecha = re.compile(r'^\d{2}/\d{2}$')
        
        for page in pdf.pages:
            if "DETALLE DE MOVIMIENTOS" in (page.extract_text() or ""):
                table = page.extract_table({
                    "vertical_strategy": "text",
                    "horizontal_strategy": "text",
                    "snap_y_tolerance": 4
                })
                if table:
                    for row in table:
                        if row[0] and patron_fecha.match(str(row[0])):
                            monto = limpiar_monto(row[4])
                            transacciones.append({
                                'FECHA': row[0],
                                'VALOR': monto,
                                'DESCRIPCION': " ".join(str(row[2]).split())
                            })

        df = pd.DataFrame(transacciones)

        # --- 4. TRANSFORMACIONES Y DICCIONARIO FINAL ---
        if not df.empty:
            df.rename(columns={'VALOR': 'MONTO'}, inplace=True)

        flat_meta = {
            'EMPRESA':             meta_raw['EMPRESA'],
            'NUM_TARJETA':         meta_raw['NUM_TARJETA'],
            'FECHA_EMISION':       meta_raw['FECHA_EMISION'],
            'FECHA_MAX_PAGO':      meta_raw['FECHA_MAX_PAGO'],
            'saldo_anterior':      meta_raw['SALDO_ANTERIOR'],
            'subtotal_pagado':     meta_raw['SUBTOTAL_PAGADO'],
            'total_a_pagar':       meta_raw['TOTAL_A_PAGAR'],
            'minimo_a_pagar':      meta_raw['MINIMO_A_PAGAR'],
            'total_consumo':       meta_raw['TOTAL_CONSUMO'],
            'num_transacciones':   len(df),
            'fecha_min':           df['FECHA'].min() if not df.empty else None,
            'fecha_max':           df['FECHA'].max() if not df.empty else None,
            'total_mes':           df['MONTO'].sum() if not df.empty else 0,
            'total_a_pagar_despues': meta_raw['TOTAL_A_PAGAR'] + (df['MONTO'].sum() if not df.empty else 0),
            'source_file':         file_name,
        }

        return df, flat_meta

In [4]:
import pdfplumber
import pandas as pd
import re
import os

def procesar_estado_pichincha_completo(file_path):
    file_name = os.path.basename(file_path)
    
    with pdfplumber.open(file_path) as pdf:
        # Extraemos el texto completo para el parsing de metadata
        texto_completo = "\n".join([p.extract_text() for p in pdf.pages if p.extract_text()])
        
        # --- 1. FUNCIÓN INTERNA DE LIMPIEZA ---
        def limpiar_monto(texto):
            if not texto: return 0.0
            # Normalización: 1.234,56 -> 1234.56
            num = texto.replace('.', '').replace(',', '.')
            if num.endswith('-'):
                num = '-' + num.replace('-', '')
            try:
                return float(num)
            except ValueError:
                return 0.0

        # --- 2. EXTRACCIÓN DE METADATA CON REGEX ---
        # Buscamos los valores clave en el texto del PDF
        meta_raw = {
            'EMPRESA':          "BANCO PICHINCHA",
            'NUM_TARJETA':      re.search(r'(\d{4}X+5647)', texto_completo).group(1) if re.search(r'(\d{4}X+5647)', texto_completo) else "N/A",
            'FECHA_EMISION':    re.search(r'FECHA DE EMISIÓN:\s+([\w\s]+)', texto_completo).group(1).strip() if re.search(r'FECHA DE EMISIÓN:\s+([\w\s]+)', texto_completo) else None,
            'FECHA_MAX_PAGO':   re.search(r'FECHA MÁXIMA DE PAGO SIN RECARGOS\s+([\w\s]+)', texto_completo).group(1).strip() if re.search(r'FECHA MÁXIMA DE PAGO SIN RECARGOS\s+([\w\s]+)', texto_completo) else None,
            'SALDO_ANTERIOR':   limpiar_monto(re.search(r'SALDO ANTERIOR\s+([\d,.]+)', texto_completo).group(1)) if re.search(r'SALDO ANTERIOR\s+([\d,.]+)', texto_completo) else 0.0,
            'SUBTOTAL_PAGADO':  limpiar_monto(re.search(r'SUBTOTAL PAGOS\s+([\d,.]+)', texto_completo).group(1)) if re.search(r'SUBTOTAL PAGOS\s+([\d,.]+)', texto_completo) else 0.0,
            'TOTAL_A_PAGAR':    limpiar_monto(re.search(r'TOTAL A PAGAR\s+([\d,.]+)', texto_completo).group(1)) if re.search(r'TOTAL A PAGAR\s+([\d,.]+)', texto_completo) else 0.0,
            'MINIMO_A_PAGAR':   limpiar_monto(re.search(r'MINIMO A PAGAR\s+([\d,.]+)', texto_completo).group(1)) if re.search(r'MINIMO A PAGAR\s+([\d,.]+)', texto_completo) else 0.0,
            'TOTAL_CONSUMO':    limpiar_monto(re.search(r'TOTAL CONSUMOS MES\s+([\d,.]+)', texto_completo).group(1)) if re.search(r'TOTAL CONSUMOS MES\s+([\d,.]+)', texto_completo) else 0.0,
        }

        # --- 3. EXTRACCIÓN DE MOVIMIENTOS ---
        transacciones = []
        patron_fecha = re.compile(r'^\d{2}/\d{2}$')
        
        for page in pdf.pages:
            if "DETALLE DE MOVIMIENTOS" in (page.extract_text() or ""):
                table = page.extract_table({
                    "vertical_strategy": "text",
                    "horizontal_strategy": "text",
                    "snap_y_tolerance": 4
                })
                if table:
                    for row in table:
                        if row[0] and patron_fecha.match(str(row[0])):
                            monto = limpiar_monto(row[4])
                            transacciones.append({
                                'FECHA': row[0],
                                'VALOR': monto,
                                'DESCRIPCION': " ".join(str(row[2]).split())
                            })

        df = pd.DataFrame(transacciones)

        # --- 4. TRANSFORMACIONES Y DICCIONARIO FINAL ---
        if not df.empty:
            df.rename(columns={'VALOR': 'MONTO'}, inplace=True)

        flat_meta = {
            'EMPRESA':             meta_raw['EMPRESA'],
            'NUM_TARJETA':         meta_raw['NUM_TARJETA'],
            'FECHA_EMISION':       meta_raw['FECHA_EMISION'],
            'FECHA_MAX_PAGO':      meta_raw['FECHA_MAX_PAGO'],
            'saldo_anterior':      meta_raw['SALDO_ANTERIOR'],
            'subtotal_pagado':     meta_raw['SUBTOTAL_PAGADO'],
            'total_a_pagar':       meta_raw['TOTAL_A_PAGAR'],
            'minimo_a_pagar':      meta_raw['MINIMO_A_PAGAR'],
            'total_consumo':       meta_raw['TOTAL_CONSUMO'],
            'num_transacciones':   len(df),
            'fecha_min':           df['FECHA'].min() if not df.empty else None,
            'fecha_max':           df['FECHA'].max() if not df.empty else None,
            'total_mes':           df['MONTO'].sum() if not df.empty else 0,
            'total_a_pagar_despues': meta_raw['TOTAL_A_PAGAR'] + (df['MONTO'].sum() if not df.empty else 0),
            'source_file':         file_name,
        }

        return df, flat_meta

In [1]:
%load_ext autoreload
%autoreload 2
from contabilidad.backend.services.credit_card.pdf_reader import get_credit_card_data_from_pdf

In [40]:
path ="/home/sebas/dev/projects/ContabilidadPersonal/data/nuevos/tarjeta/Marzo26.pdf"
get_credit_card_data_from_pdf(path)

TEXTO
['TARJETA MASTERCARD', 'Estado de Cuenta', 'DETALLE DE MOVIMIENTOS', 'Fecha Referencia Descripción Operación Valores Diferido', 'SALDO ANTERIOR 282,28', '23/02 SU PAGO "MUCHAS GRACIAS" 33,85-', '27/02 353996 SU PAGO "MUCHAS GRACIAS" 248,43-', 'SUBTOTAL PAGOS 282,28', '06/03 INT. FINANCIAMIENTO N/D 0,43', '19/02 8284292 IVA N/D IVA 0,03', '06/03 CONT_FIN_SOLCA_C.ROT** N/D 0,01', '19/02 8284292 TARIFA CONSUMO GASOLINERA N/D T15 0,20', 'ARCENTALES ARCINIEGA ANDRE SEB 2230-xxxx-xxxx-6211 * PRINCIPAL REEMPLAZO MASTERCARD JOVEN * ID: 1729372761', '25/02 8480498 ZURITA Y ZURITA LABORA Q CONSUMO ECU 9,50', '05/02 7681285 MENESTRAS DEL NEGRO M0 Q CONSUMO ECU 11,97', '19/02 8284292 ATIMASA ESTACION DE SE E CONSUMO ECU 34,56', '11/02 7929944 UB *EATS ECUADOR 8 CONSUMO NLD 4,64', '11/02 7929944 RET IVA SERV DIGITAL 10% N/D 0,07', '12/02 7992502 ALITAS CADILAC 2 HEMIS E CONSUMO ECU 15,75', '25/02 8521098 SUPERMAXI BICENTENARIO Q CONSUMO ECU 39,57', '27/02 8613893 DLC*UBER RIDESCU S CONSUMO CR

(    FECHA   CODIGO                           DESCRIPCION  MONTO
 0   25/02  8480498  ZURITA Y ZURITA LABORA Q CONSUMO ECU   9.50
 1   05/02  7681285  MENESTRAS DEL NEGRO M0 Q CONSUMO ECU  11.97
 2   19/02  8284292  ATIMASA ESTACION DE SE E CONSUMO ECU  34.56
 3   11/02  7929944        UB *EATS ECUADOR 8 CONSUMO NLD   4.64
 4   11/02  7929944          RET IVA SERV DIGITAL 10% N/D   0.07
 5   12/02  7992502  ALITAS CADILAC 2 HEMIS E CONSUMO ECU  15.75
 6   25/02  8521098  SUPERMAXI BICENTENARIO Q CONSUMO ECU  39.57
 7   27/02  8613893        DLC*UBER RIDESCU S CONSUMO CRI   3.99
 8   27/02  8613893         RET IVA SERV DIGITAL 100% N/D   0.60
 9   20/02  8309436        DLC*UBER EATSSWE S CONSUMO CRI   4.42
 10  20/02  8309436          RET IVA SERV DIGITAL 10% N/D   0.07
 11  22/02  8335397             SIME PORTAL Q CONSUMO ECU  16.94
 12  22/02  8377121           MCDONALD S KS G CONSUMO ECU   5.35
 13  25/02  8455428     AIRBNB * HMZ42TBNSJ L CONSUMO GBR  81.00
 14  25/02  8455428      

In [35]:
# path ="/home/sebas/Descargas/Temporal/2230XXXXXXXX5647.pdf" 
df =extraer_fino(path)
totales = extraer_totales_pichincha(path)
df

TEXTO
['TARJETA MASTERCARD', 'Estado de Cuenta', 'DETALLE DE MOVIMIENTOS', 'Fecha Referencia Descripción Operación Valores Diferido', 'SALDO ANTERIOR 282,28', '23/02 SU PAGO "MUCHAS GRACIAS" 33,85-', '27/02 353996 SU PAGO "MUCHAS GRACIAS" 248,43-', 'SUBTOTAL PAGOS 282,28', '06/03 INT. FINANCIAMIENTO N/D 0,43', '19/02 8284292 IVA N/D IVA 0,03', '06/03 CONT_FIN_SOLCA_C.ROT** N/D 0,01', '19/02 8284292 TARIFA CONSUMO GASOLINERA N/D T15 0,20', 'ARCENTALES ARCINIEGA ANDRE SEB 2230-xxxx-xxxx-6211 * PRINCIPAL REEMPLAZO MASTERCARD JOVEN * ID: 1729372761', '25/02 8480498 ZURITA Y ZURITA LABORA Q CONSUMO ECU 9,50', '05/02 7681285 MENESTRAS DEL NEGRO M0 Q CONSUMO ECU 11,97', '19/02 8284292 ATIMASA ESTACION DE SE E CONSUMO ECU 34,56', '11/02 7929944 UB *EATS ECUADOR 8 CONSUMO NLD 4,64', '11/02 7929944 RET IVA SERV DIGITAL 10% N/D 0,07', '12/02 7992502 ALITAS CADILAC 2 HEMIS E CONSUMO ECU 15,75', '25/02 8521098 SUPERMAXI BICENTENARIO Q CONSUMO ECU 39,57', '27/02 8613893 DLC*UBER RIDESCU S CONSUMO CR

,Fecha,Ref,Descripción,Monto
0,25/02,8480498,ZURITA Y ZURITA LABORA Q CONSUMO ECU,9.50
1,05/02,7681285,MENESTRAS DEL NEGRO M0 Q CONSUMO ECU,11.97
2,19/02,8284292,ATIMASA ESTACION DE SE E CONSUMO ECU,34.56
3,11/02,7929944,UB *EATS ECUADOR 8 CONSUMO NLD,4.64
4,11/02,7929944,RET IVA SERV DIGITAL 10% N/D,0.07
5,12/02,7992502,ALITAS CADILAC 2 HEMIS E CONSUMO ECU,15.75
6,25/02,8521098,SUPERMAXI BICENTENARIO Q CONSUMO ECU,39.57
7,27/02,8613893,DLC*UBER RIDESCU S CONSUMO CRI,3.99
8,27/02,8613893,RET IVA SERV DIGITAL 100% N/D,0.60
9,20/02,8309436,DLC*UBER EATSSWE S CONSUMO CRI,4.42


In [13]:
df,flat_meta =procesar_estado_pichincha_completo(path)
df

""


In [14]:
df

""


In [33]:
df.head(20)

,Fecha,Ref,Descripción,Monto
0,27/02,353996,"SU PAGO ""MUCHAS GRACIAS""",-248.43
1,19/02,8284292,IVA N/D IVA,0.03
2,19/02,8284292,TARIFA CONSUMO GASOLINERA N/D T15,0.20
3,25/02,8480498,ZURITA Y ZURITA LABORA Q CONSUMO ECU,9.50
4,05/02,7681285,MENESTRAS DEL NEGRO M0 Q CONSUMO ECU,11.97
5,19/02,8284292,ATIMASA ESTACION DE SE E CONSUMO ECU,34.56
6,11/02,7929944,UB *EATS ECUADOR 8 CONSUMO NLD,4.64
7,11/02,7929944,RET IVA SERV DIGITAL 10% N/D,0.07
8,12/02,7992502,ALITAS CADILAC 2 HEMIS E CONSUMO ECU,15.75
9,25/02,8521098,SUPERMAXI BICENTENARIO Q CONSUMO ECU,39.57
